In [2]:
import boto3
import pandas as pd
import io

BUCKET_NAME = 'ins-churn-data'
REGION      = 'us-east-1'
s3 = boto3.client('s3', region_name=REGION)

# List every object, confirms auth works and shows exact key paths
paginator = s3.get_paginator('list_objects_v2')
pages     = paginator.paginate(Bucket=BUCKET_NAME)
all_keys  = []
for page in pages:
    for obj in page.get('Contents', []):
        all_keys.append(obj['Key'])
        print(f"  {obj['Key']:<60} {obj['Size']:>10,} bytes")
print(f'Total: {len(all_keys)} objects')


  bridge_customer_agent.csv                                        23,859 bytes
  cleaned/                                                              0 bytes
  cleaned/bridge_customer_agent_cleaned.csv                        23,859 bytes
  cleaned/dim_agents_cleaned.csv                                    4,126 bytes
  cleaned/dim_customers_cleaned.csv                                91,159 bytes
  cleaned/dim_geography_cleaned.csv                                 4,783 bytes
  cleaned/dim_products_cleaned.csv                                 13,404 bytes
  cleaned/fact_policy_activity_cleaned.csv                        798,681 bytes
  dim_agents.csv                                                    4,126 bytes
  dim_customers.csv                                                91,159 bytes
  dim_geography.csv                                                 4,783 bytes
  dim_products.csv                                                 13,404 bytes
  fact_policy_activity.csv              

In [3]:
import boto3, pandas as pd, numpy as np, io, os, json, logging, pickle

logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
log = logging.getLogger(__name__)

BUCKET_NAME = 'ins-churn-data'
REGION      = 'us-east-1'
s3          = boto3.client('s3', region_name=REGION)
LOCAL_TMP   = '/tmp/ins_churn'   # only writable local path in SageMaker Notebook
os.makedirs(LOCAL_TMP, exist_ok=True)

def read_csv_from_s3(bucket, key, **kwargs):
    response = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(io.BytesIO(response['Body'].read()), **kwargs)

def write_csv_to_s3(df, bucket, key):
    buffer = io.StringIO()
    df.to_csv(buffer, index=False)
    s3.put_object(Bucket=bucket, Key=key, Body=buffer.getvalue())
    log.info('  saved -> s3://%s/%s (%d rows)', bucket, key, len(df))

def write_json_to_s3(data, bucket, key):
    s3.put_object(Bucket=bucket, Key=key,
                  Body=json.dumps(data, indent=2, default=str))

def upload_file_to_s3(local_path, bucket, key):
    s3.upload_file(local_path, bucket, key)  # for model artifacts via /tmp

def make_key(folder, filename):
    return f'{folder}/{filename}' if folder else filename

log.info('Helpers ready.')


INFO | Helpers ready.


In [ ]:
INPUT_DIR  = ''        # raw files at bucket root
OUTPUT_DIR = 'cleaned'

def clean_products(df):
    for col in ['unit_price', 'cost']:  # stored as '1,248' strings
        if col in df.columns and df[col].dtype == object:
            df[col] = pd.to_numeric(
                df[col].astype(str).str.replace(',', '', regex=False), errors='coerce'
            )
    if 'active_flag' in df.columns:  # mixed: True/'08/08/2018'/False
        df['active_flag'] = df['active_flag'].astype(str).str.strip().str.lower()\
                                             .map({'true': 1, 'false': 0})
    return df

def clean_customers(df):
    if 'status' in df.columns:  # mixed: 'Active'/'ACTIVE'/'active'
        df['status'] = df['status'].astype(str).str.strip().str.lower()
    return df

def clean_fact(df):
    for col in ['premium_amount', 'transaction_amount']:  # stored as '3,220'
        if col in df.columns and df[col].dtype == object:
            df[col] = pd.to_numeric(
                df[col].astype(str).str.replace(',', '', regex=False), errors='coerce'
            )
    return df

def clean_agents(df):
    if 'active' in df.columns:
        df['active'] = df['active'].astype(str).str.lower().map({'true':1,'false':0})
    return df

def clean_geography(df): return df
def clean_bridge(df):    return df

tables = {
    'dim_agents':            (make_key(INPUT_DIR,'dim_agents.csv'),            clean_agents),
    'dim_customers':         (make_key(INPUT_DIR,'dim_customers.csv'),         clean_customers),
    'dim_products':          (make_key(INPUT_DIR,'dim_products.csv'),          clean_products),
    'dim_geography':         (make_key(INPUT_DIR,'dim_geography.csv'),         clean_geography),
    'bridge_customer_agent': (make_key(INPUT_DIR,'bridge_customer_agent.csv'), clean_bridge),
    'fact_policy_activity':  (make_key(INPUT_DIR,'fact_policy_activity.csv'),  clean_fact),
}
for name, (key, cleaner) in tables.items():
    df = read_csv_from_s3(BUCKET_NAME, key)
    df_clean = cleaner(df)
    write_csv_to_s3(df_clean, BUCKET_NAME, make_key(OUTPUT_DIR, f'{name}_cleaned.csv'))
    log.info('  %s -> %d rows', name, len(df_clean))


In [ ]:
fact_raw = read_csv_from_s3(BUCKET_NAME, 'fact_policy_activity_cleaned.csv')

# Annualised premium, core revenue driver
fact_raw['annual_premium'] = fact_raw['premium_amount'] * (
    12 / fact_raw['policy_tenure_months'].clip(lower=1)
)

# Churn risk composite from columns that exist and have meaning
fact_raw['churn_risk'] = (
    (fact_raw['payment_delay_days'].fillna(0) / 30) +
    (fact_raw['claim_frequency'].fillna(0) * 0.5) +
    (fact_raw['customer_complaints'].fillna(0) * 0.3)
).clip(lower=0)

fact_raw['retention_prob'] = (1 / (1 + fact_raw['churn_risk'])).clip(0.2, 0.95)
fact_raw['expected_years'] = (
    fact_raw['retention_prob'] / (1 - fact_raw['retention_prob'])
).clip(upper=10)

profit_margin = (
    (fact_raw['transaction_amount'] - fact_raw['premium_amount'].fillna(0)) /
    fact_raw['transaction_amount'].replace(0, np.nan)
).clip(0.05, 0.95).fillna(0.3)

fact_raw['clv_rebuilt'] = (
    fact_raw['annual_premium'] * fact_raw['expected_years'] * profit_margin
).round(2)

print(fact_raw['clv_rebuilt'].describe().round(2))

# Replace CLV in features CSV and re-split
features = read_csv_from_s3(BUCKET_NAME, 'features/features.csv')
features['customer_lifetime_value'] = fact_raw['clv_rebuilt'].values
write_csv_to_s3(features, BUCKET_NAME, 'features/features.csv')

from sklearn.model_selection import train_test_split
df_train, df_val = train_test_split(features, test_size=0.2, random_state=42)
write_csv_to_s3(df_train, BUCKET_NAME, 'features/train.csv')
write_csv_to_s3(df_val,   BUCKET_NAME, 'features/val.csv')


In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load model via /tmp (XGBoost requires a file path for load_model)
xgb_tmp = os.path.join(LOCAL_TMP, 'xgb_model_eval.json')
s3.download_file(BUCKET_NAME, 'model_artifacts/xgb_model.json', xgb_tmp)
model = xgb.XGBRegressor()
model.load_model(xgb_tmp)

# Scaler and feature names loaded directly from S3 bytes, no file needed
scaler_eval = pickle.loads(
    s3.get_object(Bucket=BUCKET_NAME, Key='model_artifacts/scaler.pkl')['Body'].read()
)
feat_eval = json.loads(
    s3.get_object(Bucket=BUCKET_NAME, Key='model_artifacts/feature_names.json')['Body'].read()
)

val   = read_csv_from_s3(BUCKET_NAME, 'features/val.csv')
val_e = add_interaction_features(val)
X_ev, _, y_ev_raw, _, _ = prepare_xy(val_e, feature_names=feat_eval, scaler=scaler_eval)

preds_ev  = np.expm1(model.predict(X_ev))   # reverse log transform
residuals = y_ev_raw - preds_ev

mae  = mean_absolute_error(y_ev_raw, preds_ev)
rmse = np.sqrt(mean_squared_error(y_ev_raw, preds_ev))
r2   = r2_score(y_ev_raw, preds_ev)
mape = np.mean(np.abs(residuals / np.clip(y_ev_raw,1,None))) * 100
log.info('R2=%.4f  RMSE=%.2f  MAE=%.2f  MAPE=%.2f%%', r2, rmse, mae, mape)
log.info('Gate: %s', '✅ PASS' if r2 >= 0.70 and rmse <= 5000 else '❌ FAIL')

# Save outputs to S3
write_json_to_s3({'R2':round(r2,4),'RMSE':round(rmse,2),'MAE':round(mae,2),
                  'MAPE':round(mape,2)},
                 BUCKET_NAME, 'model_output/metrics.json')

importance = pd.DataFrame({'feature':feat_eval,'importance':model.feature_importances_})
importance = importance.sort_values('importance', ascending=False)
write_csv_to_s3(importance, BUCKET_NAME, 'model_output/feature_importance.csv')
display(importance.head(10))
log.info('✅ Evaluation saved to S3.')
